In [1]:
import pandas as pd

In [5]:
df_target = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/target_dataset.csv")
df_historical_return = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/historical_return.csv")
df_historical_volatility = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/historical_volatility.csv")
df_avg_volume = pd.read_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/average_trading_volume.csv")

In [6]:
print(df_target.columns.tolist())
print(df_historical_return.columns.tolist())
print(df_historical_volatility.columns.tolist())
print(df_avg_volume.columns.tolist())

['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 't1', 'Price at t1', 'Future Return', 'Target']
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 'One Year Before', 'Historical Date', 'Historical Price', 'Historical Gap', 'Historical Return']
['SimFinId', 'Fiscal Year', 'Historical Volatility']
['SimFinId', 'Fiscal Year', 'Average Trading Volume']


## Merge historical return

In [8]:
df_market = df_target.merge(df_historical_return[["SimFinId", "Fiscal Year", "Historical Return"]],
                            on=["SimFinId", "Fiscal Year"],
                            how="left",
                            validate="one_to_one"
                            )

In [10]:
print(df_market.shape)
print(df_market.columns.to_list())

(386, 12)
['SimFinId', 'Ticker', 'Fiscal Year', 'Publish Date', 't0', 'Price at t0', 'One Year Date', 't1', 'Price at t1', 'Future Return', 'Target', 'Historical Return']


In [11]:
print("Duplicate company-year rows:", df_market.duplicated(["SimFinId", "Fiscal Year"]).sum())

print("Missing Historical Return:", df_market["Historical Return"].isna().sum())

Duplicate company-year rows: 0
Missing Historical Return: 96


In [12]:
print(df_market[df_market["Historical Return"].isna()][["Ticker", "SimFinId", "Fiscal Year", "Publish Date", "t0"]].head(20))

   Ticker  SimFinId  Fiscal Year Publish Date          t0
0    AVGO    686523         2020   2020-11-18  2020-11-18
1      DE    373449         2020   2020-12-17  2020-12-17
2    ADBE     14099         2020   2021-01-15  2021-01-15
3    INTC     85652         2020   2021-01-22  2021-01-22
4     SLB    353500         2020   2021-01-27  2021-01-27
5    NFLX     83548         2020   2021-01-28  2021-01-28
6     LMT    103644         2020   2021-01-28  2021-01-28
7    META    121021         2020   2021-01-28  2021-01-28
8     AMD    191397         2020   2021-01-29  2021-01-29
9    GOOG        18         2020   2021-02-03  2021-02-03
10   AMZN     62747         2020   2021-02-03  2021-02-03
11    MMM     22219         2020   2021-02-04  2021-02-04
12  CMCSA    106366         2020   2021-02-04  2021-02-04
13    ICE    378167         2020   2021-02-04  2021-02-04
14    KMI     88392         2020   2021-02-05  2021-02-05
15   PYPL    109664         2020   2021-02-05  2021-02-05
16      F    2

In [13]:
print(df_market.groupby("Fiscal Year")["Historical Return"].apply(lambda x: x.isna().sum()))

Fiscal Year
2020    89
2021     7
2022     0
2023     0
2024     0
Name: Historical Return, dtype: int64


## Add Historical volatility

In [15]:
from numpy import ones
df_market = df_market.merge(df_historical_volatility[["SimFinId", "Fiscal Year", "Historical Volatility"]],
                            on=["SimFinId", "Fiscal Year"],
                            how="left",
                            validate="one_to_one"
                            )

In [17]:
print(df_market.shape)

print("Duplicate company-year rows:", df_market.duplicated(["SimFinId", "Fiscal Year"]).sum())

print("Missing Historical Volatility:", df_market["Historical Volatility"].isna().sum())

(386, 13)
Duplicate company-year rows: 0
Missing Historical Volatility: 96


## Add Average Trading Volume

In [25]:
df_market = df_market.merge(df_avg_volume[["SimFinId", "Fiscal Year", "Average Trading Volume"]],
                            on=["SimFinId", "Fiscal Year"],
                            how="left",
                            validate="one_to_one"
                            )

In [27]:
print(df_market.shape)

print("Duplicate company_year rows:", df_market.duplicated(["SimFinId", "Fiscal Year"]).sum())

print("Missing Average Trading Volume:", df_market["Average Trading Volume"].isna().sum())

(386, 14)
Duplicate company_year rows: 0
Missing Average Trading Volume: 96


In [29]:
print(df_market.info())

print("---- Inspect head----")
print(df_market.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   SimFinId                386 non-null    int64  
 1   Ticker                  386 non-null    object 
 2   Fiscal Year             386 non-null    int64  
 3   Publish Date            386 non-null    object 
 4   t0                      386 non-null    object 
 5   Price at t0             386 non-null    float64
 6   One Year Date           386 non-null    object 
 7   t1                      386 non-null    object 
 8   Price at t1             386 non-null    float64
 9   Future Return           386 non-null    float64
 10  Target                  386 non-null    int64  
 11  Historical Return       290 non-null    float64
 12  Historical Volatility   290 non-null    float64
 13  Average Trading Volume  290 non-null    float64
dtypes: float64(6), int64(3), object(5)
memory 

### **Save to outputs/market_features

In [30]:
df_market.to_csv("/content/drive/MyDrive/data_handling_and_machine_learning/week_03/outputs/market_features/merged_market_features.csv", index=False)